# Daniel Peña Fonseca

Use of LLMs (Claude Sonet 4.6) for this exercise:
- Exercise 6.1 Since I had never downloaded data from yfinance, it made the loading process simpler for me. Also, for selecting which stocks to study.

- Exercise 6.2 and 6.3: Since I had never worked with pandas before (that I can remember), it was really useful for handling data with this library.

- Exercise 6.4: For the section 'Measuring lengths', it was used for the 'More sophisticated' approach at the bottom of that jupyter cell. It was also used for keras.preprocessing.sequence.pad_sequences, and for noticing that in a classification problem, it is better to use the AUC metric. Although it helped with handling the data using pandas, which was a big part of the help for the exercise, the biggest advantage of using an LLM in this exercise was that it suggested using class weights, since this suggestion improved my results noticeably. It was also used to make the code cleaner.

- Exercise 6.5: Since I had worked with classes only once before this exercise, it was extensively used in this case.

# Exercise 6

# Exercise 6.1

For this exercise, we simply download the data. First, more stocks were selected (around 75), however, some of them had a different length (more days for those stocks than for the other stocks), so we kept the ones presented above (same length). The length of these is checked in Exercise 6.4, in the section 'Measuring lengths'. For the download, we simply set an initial date, a final date, and we keep prices (closing) and volumes. Using auto_adjust we make sure that we ajust for splits/dividends.

In [1]:
%pip install -q yfinance pandas numpy tensorflow

Note: you may need to restart the kernel to use updated packages.


In [257]:
import yfinance as yf
import pandas as pd
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import copy
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import Input

In [258]:


# ≈ 75 stocks
"""TICKERS = [
    # Tech
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'INTC', 'AMD', 'ORCL',
    'IBM', 'QCOM', 'TXN', 'CRM', 'ADBE', 'NFLX', 'AVGO', 'MU', 'AMAT', 'NOW',
    # Finance
    'JPM', 'BAC', 'WFC', 'GS', 'MS', 'C', 'AXP', 'BLK', 'SCHW', 'USB',
    # Healthcare
    'JNJ', 'PFE', 'UNH', 'MRK', 'ABBV', 'TMO', 'ABT', 'LLY', 'BMY', 'AMGN',
    # Consumer
    'KO', 'PEP', 'PG', 'WMT', 'COST', 'MCD', 'SBUX', 'NKE', 'HD', 'TGT',
    # Energy
    'XOM', 'CVX', 'COP', 'SLB', 'EOG',
    # Industrials
    'GE', 'BA', 'CAT', 'MMM', 'HON', 'UPS', 'RTX', 'LMT', 'DE',
    # Telecom / Media
    'VZ', 'T', 'CMCSA', 'DIS', 'NFLX',
    # Other
    'V', 'MA', 'PYPL', 'BRK-B', 'SPY', 'QQQ'
]"""
# ≈ 75 stocks
TICKERS = [
    # Tech
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'INTC', 'AMD', 'ORCL',
    'IBM', 'QCOM', 'TXN', 'CRM', 'ADBE', 'NFLX', 'AVGO', 'MU', 'AMAT', 'NOW',
    # Finance
    'JPM', 'BAC', 'WFC', 'GS', 'MS', 'C', 'AXP', 'BLK', 'SCHW', 'USB',
    # Healthcare
    'JNJ', 'PFE', 'UNH', 'MRK', 'ABBV', 'TMO', 'ABT', 'LLY', 'BMY', 'AMGN',
    # Consumer
    'KO', 'PEP', 'PG', 'WMT', 'COST', 'MCD', 'SBUX', 'NKE', 'HD', 'TGT',
    # Energy
    'XOM', 'CVX', 'COP', 'SLB', 'EOG',
]
# Remove any accidental duplicates
TICKERS = list(dict.fromkeys(TICKERS))
print(f'Total tickers: {len(TICKERS)}')

# Download data
START_DATE = '2015-01-01'
END_DATE   = None # None => today

raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True, # prices already adjusted for splits/dividends
    progress=True
)

# We only keep adj close (which is called Close when auto_adjust=True) and Volume
close  = raw['Close'].copy()
volume = raw['Volume'].copy()

print('Close shape :', close.shape)
print('Date range  :', close.index[0].date(), '→', close.index[-1].date())

Total tickers: 55


[*********************100%***********************]  55 of 55 completed


Close shape : (2810, 55)
Date range  : 2015-01-02 → 2026-03-06


# Exercise 6.2 & 6.3

Since we will be splitting the data into three different sets (training/validation/test) using the dates, we will need to handle the 'date'; also, we will do predictions for different stocks, so we also consider the 'ticker'. Finally, since we work with the closing prices and the volume, we will consider 'adj_close' and 'volume'.

In [259]:
# We build per-ticker long-format data frame
def build_df(close_df, volume_df, threshold=0.03):
    frames = []
    for ticker in close_df.columns:
        s_close  = close_df[ticker].dropna()
        s_volume = volume_df[ticker].reindex(s_close.index)

        df = pd.DataFrame({
            'date'      : s_close.index.strftime('%Y%m%d'), # we format the sate as string: '20230312' for example
            'ticker'    : ticker,
            'adj_close' : s_close.values,
            'volume'    : s_volume.values
        })

        # Keep as pandas Series (no .values) so .apply() works
        next_close            = s_close.shift(-1) # next day's price. This will help us see if there is an increment, and if so, how big. So we can see whether we should have invested the previous day or not.
        percentage_difference = (next_close / s_close) - 1 # percentage difference from previous day to next day's closing price

        df['target'] = percentage_difference.apply(
            lambda x: float('nan') if pd.isna(x) else int(x >= threshold) # if there is no 'tomorrow', as happens with the last date, we consider Nan. If there is, we put a 0 if the increment is below 3%, 1 otherwise
        ).values

        frames.append(df)

    full = pd.concat(frames, ignore_index=True) # we stack the tickers
    full['date'] = full['date'].astype(str) # we make sure every day stays as string
    return full

full_df = build_df(close, volume)
print(f'Full data frame shape: {full_df.shape}')
#print('Head of full data frame:')
full_df.head()

Full data frame shape: (154550, 5)


,date,ticker,adj_close,volume,target
0,20150102,AAPL,24.214895,212818400,0.0
1,20150105,AAPL,23.532717,257142000,0.0
2,20150106,AAPL,23.534937,263188400,0.0
3,20150107,AAPL,23.864944,160423600,1.0
4,20150108,AAPL,24.781891,237458000,0.0


In [260]:
# Split into train/validation/test according to date
train_df = full_df[full_df['date'] <= '20231231'].copy()
val_df   = full_df[(full_df['date'] >= '20240101') & (full_df['date'] <= '20241231')].copy()
test_df  = full_df[full_df['date'] >= '20250101'].copy()

print(f'Train : {train_df.shape}  | dates {train_df["date"].min()} → {train_df["date"].max()}  |  target=1: {(train_df['target'].mean() * 100):.1f}%')
print(f'Val   : {val_df.shape}  | dates {val_df["date"].min()} → {val_df["date"].max()}  |  target=1: {(val_df['target'].mean() * 100):.1f}%')
print(f'Test  : {test_df.shape}  | dates {test_df["date"].min()} → {test_df["date"].max()}  |  target=1: {(test_df['target'].mean() * 100):.1f}%')

Train : (124520, 5)  | dates 20150102 → 20231229  |  target=1: 5.0%
Val   : (13860, 5)  | dates 20240102 → 20241231  |  target=1: 4.0%
Test  : (16170, 5)  | dates 20250102 → 20260306  |  target=1: 5.9%


# Exercise 6.4

## Measuring lengths

In [261]:
# We measure the length of each stock for each data set
for k in range(3):
    
    if k==0:
        data_frame = train_df
        print('Training data.')
    if k==1:
        data_frame = val_df
        print('\n\nValidation data.')
    if k==2:
        data_frame = test_df
        print('\n\nTest data.')

    # Detecting days per ticker in first case
    data_length = len(data_frame["date"])
    for i in range(data_length):
        a = data_frame["ticker"].iloc[i]
        b = data_frame["ticker"].iloc[i+1]
        if a != b:
            days_per_ticker_1 = i+1
            break

    print(f'Days per ticker: {days_per_ticker_1}')

    # Detecting days per ticker in all cases
    multiple_of_days_per_ticker_list = []
    remainder_of_days_per_ticker_list = []
    number = range(data_length)
    for j in range(len(TICKERS)):
        for i in number:
            try:
                a = data_frame["ticker"].iloc[i]
                b = data_frame["ticker"].iloc[i+1]
                if a != b:
                    days_per_ticker = i+1
                    #print(train_df["ticker"].iloc[i])
                    if j==len(TICKERS)-2:
                        penultimate_index=i
                    multiple_of_days_per_ticker_list.append(days_per_ticker%days_per_ticker_1)
                    remainder_of_days_per_ticker_list.append(days_per_ticker//days_per_ticker_1)
                    break
            except IndexError:
                current_length = i+1#-penultimate_index
                multiple_of_days_per_ticker_list.append(current_length%days_per_ticker_1)
                remainder_of_days_per_ticker_list.append(current_length//days_per_ticker_1)
                #print(f'Last length: {current_length}')

        number = range(i+1, data_length)
    
    #print(multiple_of_days_per_ticker_list)
    #print(remainder_of_days_per_ticker_list)
    for i in range(len(multiple_of_days_per_ticker_list)):
        if multiple_of_days_per_ticker_list[i] != 0:
            print('Not all tickers have the same length')
            break
        if remainder_of_days_per_ticker_list[i] != i+1:
            print('Not all tickers have the same length')
            break
    # Since the message is not printed, all tickers selected have the same length: days_per_ticker_1=2264


    # More sophisticated:
    # We group the data fram be ticker and see what sizes do these groups have (number of rows = number of time frames)
    rows_per_ticker = data_frame.groupby('ticker').size()
    # print(rows_per_ticker)
    print(f'Min rows: {rows_per_ticker.min()}, Max rows: {rows_per_ticker.max()}') 

    # We check that the last row in test set has a NaN in target
    print('\nLast 2 rows of AAPL:')
    print(data_frame[data_frame['ticker'] == TICKERS[0]].tail(2))

Training data.
Days per ticker: 2264
Min rows: 2264, Max rows: 2264

Last 2 rows of AAPL:
          date ticker   adj_close    volume  target
2262  20231228   AAPL  191.589661  34049900     0.0
2263  20231229   AAPL  190.550476  42672100     0.0


Validation data.
Days per ticker: 252
Min rows: 252, Max rows: 252

Last 2 rows of AAPL:
          date ticker   adj_close    volume  target
2514  20241230   AAPL  250.829788  35557500     0.0
2515  20241231   AAPL  249.059464  39480700     0.0


Test data.
Days per ticker: 294
Min rows: 294, Max rows: 294

Last 2 rows of AAPL:
          date ticker   adj_close    volume  target
2808  20260305   AAPL  260.290009  49658600     0.0
2809  20260306   AAPL  257.459991  41094000     NaN




In this case, we define an LSTM that uses the previous window_size values for the prediction of the next value and follow the instructions from the statement. We will use early stopping and reduce the learning rate progressively.

Important consideration. Although using z-score normalization is a useful approach, since this practice helps with preventing the model from learning what the behavior is at specific values of volume, rather focusing on the change with respect to average; this procedure cannot be followed for the prices, since we would lose the percentage difference. If we have one value being 1, and next value being 1.05, when we use z-score normalization, we will likely lose the proportion 1.02/1=1.02; i.e. imagine for that window the mean was 0.99, and the standard deviation was 1. Applying z-score normalization:
$$
1.05 \rightarrow \frac{1.02-0.99}{1}=0.03; \text{ while } 1.00 \rightarrow \frac{1.00-0.99}{1}=0.01; \Rightarrow \frac{1.02}{1.00} \neq \frac{0.03}{0.01}
$$
We can see how the proportions are not maintained, which is a feature that must be considered for the prices.
To maintain proportions, we should divide by a certain number, like the first value of the window, or the average value of the window.






In [262]:

# Data preparation
THRESHOLD   = 0.5 # initial threshold. we will tune on the validation set later
FEATURES    = ['adj_close', 'volume']
WARMUP      = 10
WINDOW_SIZE = 40

def make_sequences(df, warmup=WARMUP, window_size=WINDOW_SIZE):
    "This function turns the data into windows so that the LSTM can predict."
    X_list, y_list = [], []

    # We separate the data frame by tickets (in the order they are, no alphabetical order)
    for ticker, grp in df.groupby('ticker', sort=False):
        grp = grp.sort_values('date').reset_index(drop=True)
        grp = grp.dropna(subset=['target']) # We remove the last row, the one with NaN value

        # We store data in variables
        prices  = grp['adj_close'].values.astype(np.float32)
        vols    = grp['volume'].values.astype(np.float32)
        targets = grp['target'].values.astype(np.float32)

        feats = np.stack([prices, vols], axis=1)  # (number_of_days, 2). (:, 0) is prices. (:, 1) is volume. 

        for t in range(len(feats)):
            # For values below warmup, I don't do anything
            if t < warmup:
                continue

            # The starting point for the window is 0 when t<m (since range starts at t=0), and t-window_size when t>m
            start   = max(0, t - window_size)
            window  = feats[start:t].copy() # (window_size, 2) since we have prices and volume

            # We normalize per window to avoid the LSTM from learning the scale
            # Prices: divide by the first price in the window → starts at 1.0
            
            window[:, 0] = window[:, 0] / (window[0, 0] + 1e-8)

            # Volume: z-score within the window
            v_mean = window[:, 1].mean()
            v_std  = window[:, 1].std() + 1e-8
            window[:, 1] = (window[:, 1] - v_mean) / v_std

            X_list.append(window)
            y_list.append(targets[t])

    X_padded = keras.preprocessing.sequence.pad_sequences(
        X_list, maxlen=window_size, dtype='float32', padding='pre', value=0.0
    )
    y = np.array(y_list, dtype=np.float32)
    return X_padded, y



# LSTM: model definition
# We want to use the AUC, since it is more representative when trying to separate classes
def build_lstm(window=WINDOW_SIZE, n_features=2, lstm_units=64, dropout=0.3, lr=1e-3):
    inp = keras.Input(shape=(window, n_features))
    x   = layers.LSTM(lstm_units, return_sequences=True)(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.LSTM(lstm_units // 2)(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model



In [298]:
# LSTM training with early stopping
# The LSTM also includes a reduction on the learning rate to avoid plateaus
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', mode = 'max', patience=10, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5
    )
]

# After trying with different values, the best results were obtained with a window of size 40
WINDOW_SIZE = 40

# Since the weights are initialized randomly, we want to keep the seed to make sure next time the code is run, the initial weights are the same,
# and the code runs as it does now (same seed)
print(f'Window size: {WINDOW_SIZE}')
seed = 55
print(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

model = build_lstm(window=WINDOW_SIZE, n_features=2, lstm_units=64, dropout=0.3, lr=1e-3)
model.summary()

# Turn data set values into window sequences so that the LSTM can predict
X_train, y_train = make_sequences(train_df, warmup=WARMUP, window_size=WINDOW_SIZE)
X_val,   y_val   = make_sequences(val_df, warmup=WARMUP, window_size=WINDOW_SIZE)

# To avoid an imbalance, we can use class weights.
# This is very important when we have class imbalance. If 95% of data are 0s, and 5% are 1s, a model that always predicts 0 will have high
# accuracy. This way, we want to make sure that predicting those few 1s as 1s is as important as predicting all others as 0s (that would be
# if we assigned class_weight neg/pos for 1s). 
# However, we will soften it a little bit (neg/pos)/2, since neg/pos gave worse results, probably because the punishment was too high.
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

class_weight = {
    0: 1.0,
    1: (neg / pos)/1.1
}

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)


Window size: 40
55


Model: "functional_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_32 (InputLayer)     │ (None, 40, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_66 (LSTM)                  │ (None, 40, 64)         │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_74 (Dropout)            │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_67 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_75 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,601 (115.63 KB)

 Trainable params: 29,601 (115.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
485/485 ━━━━━━━━━━━━━━━━━━━━ 17s 33ms/step - accuracy: 0.7671 - auc: 0.5753 - loss: 1.2316 - val_accuracy: 0.8485 - val_auc: 0.5514 - val_loss: 0.5902 - learning_rate: 0.0010
Epoch 2/50
485/485 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - accuracy: 0.7888 - auc: 0.6015 - loss: 1.2130 - val_accuracy: 0.8484 - val_auc: 0.5615 - val_loss: 0.5729 - learning_rate: 0.0010
Epoch 3/50
485/485 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - accuracy: 0.7854 - auc: 0.6108 - loss: 1.2080 - val_accuracy: 0.8062 - val_auc: 0.5719 - val_loss: 0.5863 - learning_rate: 0.0010
Epoch 4/50
485/485 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - accuracy: 0.7787 - auc: 0.6351 - loss: 1.1928 - val_accuracy: 0.7991 - val_auc: 0.6189 - val_loss: 0.5939 - learning_rate: 0.0010
Epoch 5/50
485/485 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - accuracy: 0.7671 - auc: 0.6593 - loss: 1.1740 - val_accuracy: 0.8073 - val_auc: 0.6310 - val_loss: 0.5657 - learning_rate: 0.0010
Epoch 6/50
485/485 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - accuracy: 0.75

In [299]:
# Predictions for validation set. Since we use softmax, val_preds will be values between 0 and 1. This is the confidence of the value.
# if val_pred at day 5 is 0.82, that means that the model is has a confidence of 82% that we should invest.
# Tuning the threshold we are modifying what is the minimum confidence for investment. 
val_preds = model.predict(X_val, batch_size=256).flatten() 



# We use the same logic that we used in make_sequences for get_prices_arrays, so that both behave the same way
def get_price_arrays(df, warmup=WARMUP, window_size=WINDOW_SIZE):
    # We save a list with current prices (curr_prices) and a list with next prices (next_prices)
    curr_prices, next_prices = [], []

    for ticker, grp in df.groupby('ticker', sort=False):
        grp    = grp.sort_values('date').reset_index(drop=True)
        grp    = grp.dropna(subset=['target'])
        prices = grp['adj_close'].values.astype(np.float32)

        for t in range(len(prices)):
            # We ignore the first values (warmup)
            if t < warmup:
                continue
            # After dropna, the last row's "next price" is the first
            # row that was dropped — reconstruct it from the original df
            curr_prices.append(prices[t])
            if t + 1 < len(prices):
                next_prices.append(prices[t + 1])
            else:
                next_prices.append(np.nan)  # last row has Nan, since we have no information about the next value

    return np.array(curr_prices, dtype=np.float32), \
           np.array(next_prices, dtype=np.float32)

val_curr, val_next = get_price_arrays(val_df) # Formatting
# val_curr: [day_1, dat_2, ..., day_99, day_100]
# val_next: [day_2, dat_3, ..., day_100, Nan]
# We can study what is the average behavior between day and next-day values (for that, we want to capture from day_2 to day_100)



# We compute the profit if we bought every day. We exclude warmup days so we can compare results with LSTM's predictions
valid_mask   = np.isfinite(val_next) # We avoid computing the last values of val_next which are Nan for each ticker
daily_returns = val_next[valid_mask] / val_curr[valid_mask] - 1
print(f"Daily return stats — mean: {daily_returns.mean():.8f}  "
      f"std: {daily_returns.std():.4f}  "
      f"min: {daily_returns.min():.4f}  "
      f"max: {daily_returns.max():.4f}")



# Profit evaluation
def evaluate_profit(preds, prices_curr, prices_next, threshold, n_stocks, n_steps):
    """
    Taking into account the days where the model decided to invest, we compute the profit: \sum_t [(p(t+1)/p(t)) - 1], where
    t goes over the days t when the model wanted to invest (i.e. those where the investment confidence val_preds was geq to threshold).
    Finally, we normalize dividing by the total number of stocks under study and the number of days in the data set considered.
    """

    # Every day where confidence is above or equal to threshold, we invest in the stock unders study. Those are the t values that we will consider.
    mask = preds >= threshold

    # We could have a model where there were no days where the confidence was enough to invest.
    if mask.sum() == 0:
        return 0.0, 0

    # Only considering investing days (considered by model) for profit evaluation
    current = prices_curr[mask]
    next = prices_next[mask]

    valid   = np.isfinite(next) # We avoid last Nan
    profits = next[valid] / current[valid] - 1

    # Just in case
    if len(profits) == 0:
        return 0.0, 0

    normalised = profits.sum() / (n_stocks * n_steps) # Normalization (dividing by number of stocks and number of days of the data set under study)

    # We return the normalised profit and the number of trades. The lower the threshold, the higher the number of trades.
    return float(normalised), int(valid.sum())


N_STOCKS = len(TICKERS) # number of stocks
M_VAL    = (val_df.groupby('ticker')['date'].count().mean()-WARMUP-1) # In this case, we do not need to do the mean, since all the stocks have the same length
                                                                    # but I leave it just in case. The -1 is for the last Nan not to be considered


# Threshold sweep
best_threshold, best_profit = 0.5, -np.inf
results = []

for threshold in np.arange(0.00, 0.95, 0.01):
    # Profit and number of trades in validation set given a certain threshold
    profit, n_trades = evaluate_profit(
        val_preds, val_curr, val_next,
        threshold, N_STOCKS, M_VAL
    )
    results.append((round(threshold, 2), profit, n_trades))

    # Threshold for which the highest profit is obtained (in validation set)
    if profit > best_profit:
        best_profit, best_threshold = profit, round(threshold, 2)

print(f"\nBest threshold (maximum validation profit): {best_threshold:.2f} -> avg profit/day = {best_profit:.8f}")
print(f"\n{'Threshold':>10}  {'Avg profit/day':>16}  {'# trades':>10}  {'Avg return per trade':>12}")

for threshold, p, n in results:
    # The lower the threshold, the more the trades. We ca study the avg return per trade so see how 'benefitial' is each trade
    avg_return = (p * N_STOCKS * M_VAL / n) if n > 0 else float('nan')
    print(f"{threshold:>10.2f}  {p:>16.8f}  {n:>10}  {avg_return:>11.4%}")

 3/52 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step

<>:53: SyntaxWarning: invalid escape sequence '\s'
<>:53: SyntaxWarning: invalid escape sequence '\s'
/var/folders/77/t6fnxt9n4sj4t0n_387gtnxw0000gn/T/ipykernel_7878/3021020821.py:53: SyntaxWarning: invalid escape sequence '\s'
  Taking into account the days where the model decided to invest, we compute the profit: \sum_t [(p(t+1)/p(t)) - 1], where


52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Daily return stats — mean: 0.00089356  std: 0.0195  min: -0.2606  max: 0.2450

Best threshold (maximum validation profit): 0.13 -> avg profit/day = 0.00092813

 Threshold    Avg profit/day    # trades  Avg return per trade
      0.00        0.00089356       13255      0.0894%
      0.01        0.00089356       13255      0.0894%
      0.02        0.00089356       13255      0.0894%
      0.03        0.00089356       13255      0.0894%
      0.04        0.00089356       13255      0.0894%
      0.05        0.00089247       13240      0.0893%
      0.06        0.00088798       13201      0.0892%
      0.07        0.00088488       13116      0.0894%
      0.08        0.00089138       12986      0.0910%
      0.09        0.00090074       12798      0.0933%
      0.10        0.00089173       12572      0.0940%
      0.11        0.00090281       12312      0.0972%
      0.12        0.00092489       12019      0.1020%
      0.13        0.00092813       

The presented results, and therefore the analysis here presented are nooticeably dependent on the class weights that we set for our model. Using a class weight of 19/1.3 for prediction of '1' (and class weight q for prediction of '0'), the following results are obtained.

The average profit per day is computed as follows:
$$
\frac{1}{N\times M}\sum_{t}\left(\frac{p(t+1)}{p(t)}-1\right)
$$
Where $N$ is the number of stocks considered, and $M$ is the number of days that we are considering with the current dataset. Here, $t$ goes over the days where the model predicts that an investment should be made; i.e. when the output of the model for a certain day is above the threshold $\theta$.

Analyzing the results, one can see how, when the threshold is $\theta=0$, the average profit per day matches the 'daily return', indicating that the code is working properly. When the threshold is $\theta=0$, every output of the LSTM model will lead to a prediction, since every output is $\geq 0$.

Although most of the values are below the 'daily return' (avg. profit per day if we invested all days), with a threshold of $\theta = 0.13$, we are slightly above this value. The 'daily return' is $0.00089356$, while $\theta = 0.13$ yields an avg. profit per day of $0.00092813$; i.e. $3.457\cdot 10^{-5}$ above the 'daily return'.

Note that, the higher the threshold, the less the number of trades, since the model will unlikely be that confident about investing. In some cases, an avg. profit below 0 (loss) is obtained, indicating that even when the model is that confident about predicting, it might not be appropriate to do so.

In [301]:
# We evaluate on test set
X_test, y_test = make_sequences(test_df, warmup=WARMUP, window_size=WINDOW_SIZE)

# All shapes. We can also see again the percentage of labels that are 1 in each case
print(f'Train : X={X_train.shape}  pos-rate={y_train.mean():.3f}')
print(f'Val   : X={X_val.shape}    pos-rate={y_val.mean():.3f}')
print(f'Test  : X={X_test.shape}   pos-rate={y_test.mean():.3f}')


test_preds           = model.predict(X_test, batch_size=256).flatten() # Confidence of investment between 0 and 1 for each day
test_curr, test_next = get_price_arrays(test_df) # Current prices, Next-day proces

N_STOCKS = len(TICKERS)
M_TEST   = test_df['date'].nunique() - WARMUP - 1 # We should not consider the warmup nor the last nan

test_profit, n_trades = evaluate_profit(
    test_preds, test_curr, test_next,
    best_threshold, N_STOCKS, M_TEST
)


# If we bought every day
valid_test_mask = np.isfinite(test_next)
daily_test_return = test_next[valid_test_mask] / test_curr[valid_test_mask] - 1
daily_test_return_normalized = daily_test_return.sum() / (N_STOCKS * M_TEST)

print(f'Buy-everyday:   avg. profit/day (all predictions): {daily_test_return_normalized:.7f}')
print(f'LSTM:  avg. profit/day (threshold={best_threshold:.2f}): {test_profit:.7f}  ({n_trades} trades)')

# per day

mask           = test_preds >= best_threshold
pred_1_correct = ((test_preds >= best_threshold) & (y_test == 1)).sum() # Number of times the model predicted 1 and it was 1
pred_1_total   = (test_preds >= best_threshold).sum() # Number of times when the model predicted 1.
precision      = pred_1_correct / (pred_1_total + 1e-8) # Fraction of times that the label is 1 when the model predicts 1.

print(f'\nPrecision (pred=1 & true=1) / pred=1 : {precision:.3f}')
print(f'Fraction of days where prediction made: {mask.mean():.3f}')

Train : X=(123970, 40, 2)  pos-rate=0.050
Val   : X=(13310, 40, 2)    pos-rate=0.041
Test  : X=(15565, 40, 2)   pos-rate=0.059
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Buy-everyday:   avg. profit/day (all predictions): 0.000736194546334
LSTM:  avg. profit/day (threshold=0.13): 0.000793699757196  (14521 trades)

Precision (pred=1 & true=1) / pred=1 : 0.062
Fraction of days where prediction made: 0.936



The behavior presented here is similar to that observed for the validation data. Compared to the strategy of buying every day, the LSTM predictions are only slightly better. 

Another important characteristic can be observed analyzing the amount of times that the model predicts that an investment should be made.
Given the small threshold, $\theta=0.13$, with only 13% of investment confidence, the LSTM will invest. Thus, it is not surprising, that most of the times that we should have invested (true label = 1), the model predicted so: $93.6%$

However, only $\approx 6%$ of times, we invest following our model's approach, we will be earning 3% profit in the next day, given that:
$$
\frac{\text{Number of days the model predicted 1, and the true label was 1}}{\text{Number of days the model predicted 1}}\approx0.059
$$

It would be interesting to study what results we would obtain if we were looking for an increase in the stock price below $3%$, since maybe, in those cases, better result would be obtained. It would also be interesting to study how results would change when trying multiple different values of LSTM size, window size, etc.; i.e. a bigger hyperparameter search.



# Exercise 6.5

In this exercise, we use an evolutionary algorithm for stock market prediction. The structure will be Population -> Rule system -> Rule group -> Rule.

- Starting from the smallest component of the algorithm, we have the rule. A rule here will have an index, an operator, and a threshold. Rules only perform a simple operation: is the i-th feature greater/smaller than the threshold; where 'i' is the index, and whether we use bigger or smaller will be determined by the operator that we have in the rule. Therefore, a rule's output will be True if the condition is satisfied, and False if it is not. 

- Rules are grouped in, what we call, rule groups. Rule groups are just a list of rules, each of them with an index, an operator and a threshold; and as rules do, rule groups only have two possible outputs: True or False. For a rule group to output True, all the rules in the group must output True; otherwise, it outputs False. Thus, rule groups implement the AND logic, since they need all the rules in the group to output True for them to also fire True.

- Rule groups are grouped in, what we call, rule systems. Rule systems are just a list of rule groups, where each rule group fires True or False, depending on the day and the stock. While rule groups used the AND logic, rule systems use the OR logic: inside a rule system, if one rule group suggests investing (i.e. it outputs True), the rule system will output True. In other words, given a certain model (rule system), it is enough if one of the rule groups from the system outputs True for the system to also output True (invest).

- Finally, we have the population. The population is formed by individuals that will be the different rule systems; i.e. the different models. With this algorithm we generate offspring from parents, we mutate, and we use elitism to maintain the best performing individuals.

Determining the best performing individual (best rule system), consists on finding what is the best number of rule groups that our system should have, what are the the best number of rules for each rule group... Also, we want to determine what are the best indices, operators, and thresholds that should be used for each of the rules in our system. All in all, we are trying to find the maximum in a function of multiple variables. However, the number of combinations of indices, operators, thresholds, groupings of rules... is really large, making this problem suitable for using an evolutionary algorithm.

In [245]:
import numpy as np
import random
from copy import deepcopy


# Feature extraction using the last lookback values

LOOKBACK = 20   # days of history available at prediction time

# we extract the features
def extract_features(df, lookback=LOOKBACK, warmup=WARMUP):
    """
    Returns:
        features : np.ndarray  shape (N_samples, n_features)
        targets  : np.ndarray  shape (N_samples,)  0/1
        price_curr: np.ndarray shape (N_samples,)
        price_next: np.ndarray shape (N_samples,)

    Feature vector for each time step t (using days t-lookback … t):
        price ratios : close[t-k] / close[t]  for k in 1..lookback
        volume ratios: volume[t-k] / volume[t] for k in 1..lookback
    Total features = 2 * lookback
    """
    feat_list, tgt_list, curr_list, next_list = [], [], [], []

    for ticker, grp in df.groupby('ticker', sort=False):
        grp    = grp.sort_values('date').reset_index(drop=True) # chronological order
        grp    = grp.dropna(subset=['target']) # we drop the last row (nan)
        prices = grp['adj_close'].values.astype(np.float64)
        vols   = grp['volume'].values.astype(np.float64)
        tgts   = grp['target'].values.astype(np.float32)

        for t in range(len(prices)):
            if t < warmup or t < lookback:
                continue
            if t + 1 >= len(prices):
                continue

            p0 = prices[t] + 1e-8
            v0 = vols[t]   + 1e-8

            p_ratios = np.array([prices[t - k] / p0 for k in range(1, lookback + 1)]) # using lookback, we check: price from  yesterday/price from today. price from two days ago/price today
            v_ratios = np.array([vols[t - k]   / v0 for k in range(1, lookback + 1)]) # same for price
            feat     = np.concatenate([p_ratios, v_ratios]) # we combine the features

            feat_list.append(feat)
            tgt_list.append(tgts[t])
            curr_list.append(prices[t])
            next_list.append(prices[t + 1])

    return (np.array(feat_list,  dtype=np.float32),
            np.array(tgt_list,   dtype=np.float32),
            np.array(curr_list,  dtype=np.float32),
            np.array(next_list,  dtype=np.float32))


print("Extracting features …")
F_train, y_train_rb, pc_train, pn_train = extract_features(train_df)
F_val,   y_val_rb,   pc_val,   pn_val   = extract_features(val_df)
F_test,  y_test_rb,  pc_test,  pn_test  = extract_features(test_df)
print(f"Train {F_train.shape} | Val {F_val.shape} | Test {F_test.shape}")

N_FEAT = F_train.shape[1]   # the shape of this is 2 * LOOKBACK = 40, since we have both priices 


# We define rule and rulegroup

# The rules we set will be feature[idx] OP threshold. Since feature[idx] is division with another day, a threshold of 1.05 indicates an increase of 5%
# OP is either '<' or '>', so either above threshold or below it
# If the rule is satisfied, True fires.

class Rule:
    __slots__ = ('idx', 'op', 'threshold') # we use this to optimize memory

    def __init__(self, idx, op, threshold):
        self.idx       = int(idx) # which feature to look at. idx=0 indicates: look at price 1 day ago. idx=20 means volume ratio 1 day ago.
        self.op        = op           # '<' or '>'
        self.threshold = float(threshold) # so Rule has three things: imagine (idx=0, op='>', threshold=1.03) -> price 1 day ago was more than 3% higher than today.

    def evaluate(self, features: np.ndarray) -> np.ndarray:
        """features : (N, N_FEAT) → bool array (N,)"""
        col = features[:, self.idx] # Using the index of the rule, we see, for each day, the features from idx. This will either be a price or a volume
        if self.op == '>':
            return col > self.threshold # True where condition holds. One bool per day studied
        else:
            return col < self.threshold # True where condition holds. One bool per day studied.

    def copy(self):
        return Rule(self.idx, self.op, self.threshold) # we create a compretely independent duplicate. This is useful when we have a child that is like the parent, but we want to mutate it without mutating the parent

    def __repr__(self):
        return f"feat[{self.idx}] {self.op} {self.threshold:.4f}"


# we need all rules to fire, so we need AND. COmbining both the rules and the AND is done with this class RuleGroup
class RuleGroup:
    """A group outputs 1 <=> (if and only if) all rules in the group fire (AND logic)."""

    def __init__(self, rules=None):
        self.rules = rules if rules is not None else [] # store a list of rule objects

    def evaluate(self, features: np.ndarray) -> np.ndarray:
        """ bool array (N,)"""
        if not self.rules:
            return np.zeros(len(features), dtype=bool) # an empty rule never fires
        result = self.rules[0].evaluate(features) # we start with the first rule's True/False array
        for r in self.rules[1:]:
            result = result & r.evaluate(features) # Rule A may fire on days [1,3,5,7], rule B may fire on days [1,2,5,9]. Since we need all the conditions to be satisfied (i.e. all rules to fire), True would only appear days 1 and 5.
        return result

    def copy(self):
        return RuleGroup([r.copy() for r in self.rules]) # we copy every rule inside

# We combine the previous rule group with OR, since we can have different conditions
class RuleSystem:
    """System outputs 1 <=> (if and only if) at least one group outputs 1 (different groups go with AND, and we combine them with OR)"""

    def __init__(self, groups=None):
        self.groups = groups if groups is not None else []

    def predict(self, features: np.ndarray) -> np.ndarray:
        """→ int array (N,) of 0/1"""
        if not self.groups:
            return np.zeros(len(features), dtype=np.int8) # no groups -> never predicts 1; i.e. never suggests buying
        result = self.groups[0].evaluate(features)
        for g in self.groups[1:]:
            result = result | g.evaluate(features) # Now we use OR logic. If one of the groups suggests buying, that is enough for the model to suggests to buy, we do not need the other groups to also fire. If one day one group suggests buying, we buy.
        return result.astype(np.int8) # we turn True/False to 1/0

    def copy(self):
        return RuleSystem([g.copy() for g in self.groups])

    def __repr__(self):
        lines = []
        for i, g in enumerate(self.groups):
            lines.append(f"  Group {i}: " + " AND ".join(str(r) for r in g.rules))
        return "RuleSystem(\n" + "\n".join(lines) + "\n)"



# Fitness function
def normalised_profit(system, features, price_curr, price_next, n_stocks, n_steps):
    """
    We sum (p_{t+1}/p_t - 1) for all the days where system predicts 1 (where model suggests buying),
    and then we normalize by n_stocks * n_steps, as the statement says.
    """
    preds   = system.predict(features) # Predicting is: assign 1 (buy) or 0 for each day
    mask    = preds == 1 # since we will measure the profit, we only take into account the days were the model says that we should buy
    n_preds = mask.sum() # how many days the model considered that we should buy
    if n_preds == 0:
        return -1e6   # we give a high penalisation if the model did not suggest buying at any point
    profits = (price_next[mask] / price_curr[mask]) - 1.0 # sum of profits
    total   = profits.sum() / (n_stocks * n_steps) # we normalize
    return float(total)



# Initialisation

# the thresholds that we are going to be using for the features are completely unknown. To start with mildly reasonable thresholds, we
# will use quantile. With this, we study how are the features for the training data, and we use reasonable thresholds for that case. Therefore, 
# we will use thresholds such that, there is a trigger 5% of days, ..., until 95% of days
FEAT_QUANTILES = np.quantile(F_train, np.linspace(0.05, 0.95, 50), axis=0)  # (50, N_FEAT)

def random_rule(n_feat=N_FEAT):
    # We randomly choose the previous value that we are going to be comparing our current result with, the operation, and the threshold
    idx       = random.randint(0, n_feat - 1)
    op        = random.choice(['<', '>'])
    q_idx     = random.randint(0, len(FEAT_QUANTILES) - 1)
    threshold = float(FEAT_QUANTILES[q_idx, idx])
    return Rule(idx, op, threshold)

def random_group(min_rules=1, max_rules=4):
    # Each group will have from 1 to 4 rules
    n = random.randint(min_rules, max_rules)
    return RuleGroup([random_rule() for _ in range(n)])

def random_system(min_groups=1, max_groups=5):
    # The system has from 1 to 5 rule groups
    n = random.randint(min_groups, max_groups)
    return RuleSystem([random_group() for _ in range(n)])



# Mutation operators

def mutate_threshold(rule, sigma=0.05):
    # using a gaussian distribution, we change the threshold
    rule.threshold += random.gauss(0, sigma)

def mutate_rule(rule, n_feat=N_FEAT):
    choice = random.random()
    # With 40% probability, we modify the threshold.
    # With 30% probability, we select another index of the lookback window
    # With 30% probability, we change the OP (comparison operator)
    if choice < 0.4:
        mutate_threshold(rule, sigma=random.uniform(0.01, 0.1))
    elif choice < 0.7:
        rule.idx = random.randint(0, n_feat - 1)
        # re-sample threshold for new feature
        q_idx          = random.randint(0, len(FEAT_QUANTILES) - 1)
        rule.threshold = float(FEAT_QUANTILES[q_idx, rule.idx])
    else:
        rule.op = '<' if rule.op == '>' else '>'

def mutate_group(group, n_feat=N_FEAT):
    # mutation applied to a certain group
    # With 50% probability, we mutate a rule of the system. Which rule of the system we mutate is chosen randomly
    # With 20% probability, we add a rule if we 5 or less groups
    # With 30% probability, we remove a rule if we have 2 or more rules
    if not group.rules:
        group.rules.append(random_rule(n_feat))
        return
    choice = random.random()
    if choice < 0.5:       # mutate a random rule
        r = random.choice(group.rules)
        mutate_rule(r, n_feat)
    elif choice < 0.7:     # add a rule
        if len(group.rules) < 6:
            group.rules.append(random_rule(n_feat))
    else:                  # remove a rule
        if len(group.rules) > 1:
            group.rules.pop(random.randint(0, len(group.rules) - 1))

def mutate_system(system, n_feat=N_FEAT):
    # With 50% probability, we mutate a group. The group that we mutate is chosen randomly
    # With 20% probability, we add a new group if we have less than 8 groups
    # With 30% probability, we remove a group if we have more than 1 group
    if not system.groups:
        system.groups.append(random_group())
        return
    choice = random.random()
    if choice < 0.5:       # mutate a random group
        g = random.choice(system.groups)
        mutate_group(g, n_feat)
    elif choice < 0.7:     # add a new group
        if len(system.groups) < 8:
            system.groups.append(random_group())
    else:                  # remove a group
        if len(system.groups) > 1:
            system.groups.pop(random.randint(0, len(system.groups) - 1))



# Running
def run_evolution(
    F_tr, pc_tr, pn_tr,        # training features / prices
    F_va, pc_va, pn_va,        # validation features / prices
    n_stocks,
    n_steps_train,
    n_steps_val,
    pop_size     = 40,
    n_generations= 150,
    n_offspring  = 80,
    elite_frac   = 0.2,
    seed         = None,
    verbose      = True
):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # we initialize the population (pop_size random systems)
    population = [random_system() for _ in range(pop_size)]

    def fitness(sys):
        return normalised_profit(sys, F_tr, pc_tr, pn_tr, n_stocks, n_steps_train)

    scores     = [fitness(s) for s in population] # evaluate
    best_val   = -np.inf
    best_sys   = None

    for gen in range(n_generations):
        # we generate offspring
        offspring = []
        for _ in range(n_offspring):
            parent = random.choices(population, weights=[max(s, 0) + 1e-6 for s in scores], k=1)[0] # we use fitness-proportional selection: better scoring parents are more likely to be chosen
            child  = parent.copy()
            # we apply between 1 and 3 mutations to the child
            for _ in range(random.randint(1, 3)):
                mutate_system(child)
            offspring.append(child)

        # Score from the offspring
        off_scores = [fitness(s) for s in offspring]

        # We combine and we select
        combined        = list(zip(population + offspring, scores + off_scores))
        combined.sort(key=lambda x: x[1], reverse=True) # we sort in descending order, so that the combined variable has the best performing at the top
        n_elite         = max(1, int(elite_frac * pop_size)) # the top % always survive
        survivors       = combined[:n_elite]
        # fill rest randomly from top half if we end up with less population that pop_size
        top_half        = combined[:len(combined) // 2]
        while len(survivors) < pop_size:
            survivors.append(random.choice(top_half))
        random.shuffle(survivors)
        population = [s for s, _ in survivors]
        scores     = [sc for _, sc in survivors]

        # we check the validation score using the best system (population[0])
        current_best     = population[0]
        val_score        = normalised_profit(current_best, F_va, pc_va, pn_va,
                                             n_stocks, n_steps_val)
        if val_score > best_val:
            best_val = val_score
            best_sys = current_best.copy() # if the validation score from the best system is higher than the highest validations previously registered, we keep that one

        # we print the training process
        if verbose and (gen % 10 == 0 or gen == n_generations - 1):
            print(f"  Gen {gen:4d} | best train = {scores[0]:.6f} | best val = {best_val:.6f}"
                  f" | groups = {len(population[0].groups)}")

    return best_sys, best_val



# We perform multiple runs and pick the one that is best in validation

N_STOCKS_EVO  = len(TICKERS)
M_TRAIN_EVO   = F_train.shape[0] / N_STOCKS_EVO # days per stock
M_VAL_EVO     = F_val.shape[0]   / N_STOCKS_EVO

N_RUNS        = 5           # number of independent EA runs
POP_SIZE      = 40
N_GEN         = 150
N_OFFSPRING   = 80

all_systems   = []

for run in range(N_RUNS):
    print(f"\n{'='*55}")
    print(f"  EA Run {run + 1} / {N_RUNS}")
    print(f"{'='*55}")
    sys_best, val_profit = run_evolution(
        F_train, pc_train, pn_train,
        F_val,   pc_val,   pn_val,
        n_stocks      = N_STOCKS_EVO,
        n_steps_train = M_TRAIN_EVO,
        n_steps_val   = M_VAL_EVO,
        pop_size      = POP_SIZE,
        n_generations = N_GEN,
        n_offspring   = N_OFFSPRING,
        seed          = run * 42,
        verbose       = True
    )
    all_systems.append((sys_best, val_profit))
    print(f"  → Run {run + 1} best validation profit: {val_profit:.6f}")

# since we do N_RUNS runs, and we want to see which one yields the best validation score, we sort them in descending order and then choose the first value
all_systems.sort(key=lambda x: x[1], reverse=True)
best_system, best_val_profit = all_systems[0]
print(f"\nBest validation profit across all runs: {best_val_profit:.6f}")
print(best_system)



# Test set evaluation


M_TEST_EVO = F_test.shape[0] / N_STOCKS_EVO

test_profit_evo = normalised_profit(
    best_system, F_test, pc_test, pn_test,
    N_STOCKS_EVO, M_TEST_EVO
)

preds_test  = best_system.predict(F_test)
n_trades    = preds_test.sum()
precision   = ((preds_test == 1) & (y_test_rb == 1)).sum() / (n_trades + 1e-8) # out of all the times the model suggests buying, what fraction of those times we should have bought

# Buy-and-hold baseline
bh_evo = ((pn_test / pc_test) - 1).sum() / (N_STOCKS_EVO * M_TEST_EVO)

print(f"\nRule-based system — test profit (normalised): {test_profit_evo:.6f}  ({n_trades} trades)")
print(f"Precision (pred=1 & true=1) / pred=1        : {precision:.3f}")
print(f"Buy-and-hold baseline                        : {bh_evo:.6f}")

Extracting features …
Train (123365, 40) | Val (12705, 40) | Test (14959, 40)

  EA Run 1 / 5
  Gen    0 | best train = 0.000777 | best val = 0.000805 | groups = 5
  Gen   10 | best train = 0.000839 | best val = 0.000810 | groups = 4
  Gen   20 | best train = 0.000922 | best val = 0.000838 | groups = 4
  Gen   30 | best train = 0.000937 | best val = 0.000838 | groups = 7
  Gen   40 | best train = 0.000946 | best val = 0.000838 | groups = 7
  Gen   50 | best train = 0.000955 | best val = 0.000838 | groups = 7
  Gen   60 | best train = 0.000960 | best val = 0.000838 | groups = 8
  Gen   70 | best train = 0.000964 | best val = 0.000838 | groups = 8
  Gen   80 | best train = 0.000972 | best val = 0.000838 | groups = 7
  Gen   90 | best train = 0.000976 | best val = 0.000838 | groups = 8
  Gen  100 | best train = 0.000975 | best val = 0.000838 | groups = 8
  Gen  110 | best train = 0.000980 | best val = 0.000838 | groups = 8
  Gen  120 | best train = 0.000978 | best val = 0.000838 | groups 